# Cleaning & Validation Layer (Bronze → Silver)

Consolidated, modular notebook that cleans and validates three datasets and writes
validated **parquet** files to `data/silver/`.

**Databricks mode:** reads from bronze Delta tables produced by `ingestion_layer.ipynb`
via `spark.read.table(...).toPandas()`.

| Dataset | Reporting grain | Output file |
|---|---|---|
| ADI | `lsoa_code × year` | `data/silver/adi_clean.parquet` |
| House prices | Transaction-level | `data/silver/houseprices_clean.parquet` |
| Postcode → LSOA | Postcode (lookup) | `data/silver/postcode_clean.parquet` |

Each dataset section follows the same structure:
1. **Load** — read from the bronze Delta table (`year` already present for ADI)
2. **Validate** — quality assessment, row counts before/after each transformation, null checks, duplicate checks at the intended grain, category sanity checks
3. **Write parquet** to `data/silver/`

## 1. Imports & Configuration

In [ ]:
import os
import pandas as pd

# ── Catalog / schema (must match ingestion_layer.ipynb) ────────────────────
CATALOG = "crime_data"
SCHEMA  = "bronze"

# Silver outputs land in a Volume so parquet files live in UC, not the workspace.
SILVER_DIR = f"/Volumes/{CATALOG}/silver/outputs"
os.makedirs(SILVER_DIR, exist_ok=True)

## 2. Reusable Utilities

Pure functions shared across all three dataset sections.

In [ ]:
def standardise_columns(df, rename_map=None):
    """Strip, lowercase, spaces->underscores; then apply optional rename_map."""
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    if rename_map:
        df = df.rename(columns=rename_map)
    return df


def standardise_postcode(series):
    """Uppercase, strip, collapse internal whitespace to single space (ONS pcds format)."""
    return (
        series.str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )


def read_bronze(table):
    """Read a bronze Delta table and return a pandas DataFrame."""
    return spark.read.table(f"{CATALOG}.{SCHEMA}.{table}").toPandas()


def dedupe_aggregate(df, group_cols, agg_map):
    """Group by group_cols and aggregate with agg_map to resolve duplicates at a grain."""
    return df.groupby(group_cols, as_index=False).agg(agg_map)


def write_parquet(df, path):
    """Write df to parquet, create parent dirs if needed, and log shape."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    df.to_parquet(path, index=False)
    print(f"Written: {path}  shape={df.shape}")

## 3. Validation Helpers

Explicit checks used at every transformation stage across all datasets.

In [ ]:
def assess_quality(df, name=""):
    """Print a data-quality summary: shape, dtype, null counts/%, unique count per column."""
    label = f" [{name}]" if name else ""
    print(f"\n=== Quality Assessment{label} ===")
    print(f"Shape: {df.shape}")
    summary = pd.DataFrame({
        "dtype":    df.dtypes,
        "nulls":    df.isnull().sum(),
        "null_%":   (df.isnull().mean() * 100).round(2),
        "n_unique": df.nunique(),
    })
    display(summary)


def check_row_counts(label, before, after):
    """Print before -> after row counts with dropped count and retention %."""
    dropped = before - after
    pct_kept = round(after / before * 100, 1) if before else 0.0
    print(f"{label}: {before:,} -> {after:,}  (dropped {dropped:,}, kept {pct_kept}%)")


def check_duplicates(df, subset, name=""):
    """Assert zero duplicates on subset columns at the intended reporting grain."""
    n = df[list(subset)].duplicated().sum()
    label = f" [{name}]" if name else ""
    status = "PASS" if n == 0 else f"FAIL -- {n:,} duplicates"
    print(f"Duplicate check{label} on {list(subset)}: {status}")
    assert n == 0, f"Expected 0 duplicates on {list(subset)}, found {n}"


def check_categories(df, col, expected):
    """Print value_counts for col; assert all values are within expected set."""
    print(f"\nValue counts -- {col}:")
    display(df[col].value_counts())
    unexpected = set(df[col].dropna().unique()) - set(expected)
    status = "PASS" if not unexpected else f"FAIL -- unexpected: {unexpected}"
    print(f"Category check -- {col}: {status}")
    assert not unexpected, f"Unexpected values in '{col}': {unexpected}"

---
## Dataset 1: ADI (Area Deprivation Index)

- **Sources:** bronze Delta tables `bronze_adi_claimant`, `bronze_adi_crime`, `bronze_adi_health`
- **Reporting grain:** `lsoa_code × year`
- **Output:** `data/silver/adi_clean.parquet`
- **Columns:** `lsoa_code`, `lsoa_name`, `pop`, `year`, `claimant_rate`, `total_crime_rate`, `total_prevalence_rate`

The `year` column is already present (derived from the `ADI_<YYYY>` folder name by the ingestion layer).

### 1a. ADI-specific functions

In [ ]:
def prepare_claimant(df):
    """Standardise column names and keep the columns needed for deduplication."""
    df = standardise_columns(df, rename_map={"area_code": "lsoa_code", "area_name": "lsoa_name"})
    return df[["lsoa_code", "lsoa_name", "pop",
               "claimant_rate", "claimant_count", "year"]].copy()


def prepare_crime(df):
    """Sum every *_rate column -> total_crime_rate; keep lsoa_code, total_crime_rate, year."""
    df = standardise_columns(df, rename_map={"area_code": "lsoa_code"})
    df = df.copy()
    rate_cols = [c for c in df.columns if c.endswith("_rate")]
    df["total_crime_rate"] = df[rate_cols].sum(axis=1)
    return df[["lsoa_code", "total_crime_rate", "year"]]


def prepare_health(df):
    """Sum every *_prevalence_rate column -> total_prevalence_rate; keep lsoa_code, total_prevalence_rate, year."""
    df = standardise_columns(df, rename_map={"area_code": "lsoa_code"})
    df = df.copy()
    rate_cols = [c for c in df.columns if c.endswith("_prevalence_rate")]
    df["total_prevalence_rate"] = df[rate_cols].sum(axis=1)
    return df[["lsoa_code", "total_prevalence_rate", "year"]]

### 1b. Load from bronze Delta tables

In [ ]:
claimant = prepare_claimant(read_bronze("bronze_adi_claimant"))
crime    = prepare_crime(read_bronze("bronze_adi_crime"))
health   = prepare_health(read_bronze("bronze_adi_health"))

print(f"claimant: {len(claimant):,} rows")
print(f"crime   : {len(crime):,} rows")
print(f"health  : {len(health):,} rows")

### 1c. Quality assessment

In [ ]:
assess_quality(claimant, "claimant")
assess_quality(crime,    "crime")
assess_quality(health,   "health")

### 1d. Year coverage validation

In [ ]:
print("Rows per year -- claimant:")
display(claimant["year"].value_counts().sort_index())
print("\nRows per year -- crime:")
display(crime["year"].value_counts().sort_index())
print("\nRows per year -- health:")
display(health["year"].value_counts().sort_index())

### 1e. Deduplication -- collapse to one row per (lsoa_code, year)

In [ ]:
# Claimant: average numeric cols within (lsoa_code, year); keep first lsoa_name
claimant_clean = dedupe_aggregate(
    claimant,
    group_cols=["lsoa_code", "year"],
    agg_map={
        "lsoa_name":      "first",
        "pop":            "mean",
        "claimant_rate":  "mean",
        "claimant_count": "mean",
    },
)
claimant_clean["claimant_rate"] = claimant_clean["claimant_rate"].round(2)
claimant_clean["pop"]           = claimant_clean["pop"].round().astype(int)

check_row_counts("claimant dedup", len(claimant), len(claimant_clean))
check_duplicates(claimant_clean, ["lsoa_code", "year"], "claimant")

# Crime: average total_crime_rate within (lsoa_code, year)
crime_clean = dedupe_aggregate(
    crime,
    group_cols=["lsoa_code", "year"],
    agg_map={"total_crime_rate": "mean"},
)
crime_clean["total_crime_rate"] = crime_clean["total_crime_rate"].round(2)

check_row_counts("crime dedup", len(crime), len(crime_clean))
check_duplicates(crime_clean, ["lsoa_code", "year"], "crime")

# Health: average total_prevalence_rate within (lsoa_code, year)
health_clean = dedupe_aggregate(
    health,
    group_cols=["lsoa_code", "year"],
    agg_map={"total_prevalence_rate": "mean"},
)
health_clean["total_prevalence_rate"] = health_clean["total_prevalence_rate"].round(2)

check_row_counts("health dedup", len(health), len(health_clean))
check_duplicates(health_clean, ["lsoa_code", "year"], "health")

### 1f. Write per-metric parquets to silver

In [ ]:
write_parquet(claimant_clean, f"{SILVER_DIR}/claimant_clean.parquet")
write_parquet(crime_clean,    f"{SILVER_DIR}/crime_clean.parquet")
write_parquet(health_clean,   f"{SILVER_DIR}/health_clean.parquet")

---
## Dataset 2: House Prices (UK Land Registry PPD)

- **Source:** bronze Delta table `bronze_price_paid` (all years, all string columns)
- **Reporting grain:** Transaction-level (one row per unique transaction)
- **Output:** `data/silver/houseprices_clean.parquet`
- **Columns:** `date_of_transfer`, `postcode`, `property_type`, `ppd_category_type`, `price`

Cleaning rules per `documentation/houseprices_preprocess.md`:
- Residential property types only: D (Detached), S (Semi-detached), T (Terraced), F (Flat)
- Standard sales only: `ppd_category_type == 'A'`
- Realistic prices only: `price >= 50,000`

### 2a. Constants + prepare function

In [ ]:
# Standard 16-column Land Registry PPD schema (source files have no header)
PPD_COLUMNS = [
    "transaction_id", "price", "date_of_transfer", "postcode",
    "property_type", "old_new", "duration", "paon", "saon",
    "street", "locality", "town_city", "district", "county",
    "ppd_category_type", "record_status",
]

# Operational columns kept after cleaning (transaction_id carried for dedup, dropped after)
HP_OUTPUT_COLUMNS = [
    "transaction_id", "date_of_transfer", "postcode",
    "property_type", "ppd_category_type", "price",
]

RESIDENTIAL = {"D", "S", "T", "F"}


def prepare_price_paid(df):
    """Clean one raw per-file PPD frame. Returns (cleaned_df, stats_dict).

    stats_dict records row counts after each filter so per-filter attribution
    can be reported once all files are processed.
    """
    stats = {"raw": len(df)}

    df = df[HP_OUTPUT_COLUMNS].copy()
    df["price"]            = pd.to_numeric(df["price"], errors="coerce")
    df["date_of_transfer"] = pd.to_datetime(df["date_of_transfer"], errors="coerce")
    df["postcode"]         = standardise_postcode(df["postcode"])

    df = df[df["postcode"].notna() & (df["postcode"] != "")]
    stats["after_postcode_clean"] = len(df)

    df = df[df["property_type"].isin(RESIDENTIAL)]
    stats["after_residential_filter"] = len(df)

    df = df[df["ppd_category_type"] == "A"]
    stats["after_ppd_category_filter"] = len(df)

    df = df[df["price"] >= 50_000]
    stats["after_price_filter"] = len(df)

    return df, stats

### 2b. Load from bronze Delta table and clean per source file

In [ ]:
hp_raw = read_bronze("bronze_price_paid")

cleaned, all_stats = [], []
for src_file, group_df in hp_raw.groupby("_source_file"):
    df, stats = prepare_price_paid(group_df)
    stats["file"] = os.path.basename(src_file)
    cleaned.append(df)
    all_stats.append(stats)
    print(f"  {stats['file']}: {stats['raw']:,} raw -> {stats['after_price_filter']:,} clean")

hp_dfs, hp_stats = cleaned, all_stats

### 2c. Per-filter attribution across all files

In [ ]:
stats_df = pd.DataFrame(hp_stats).set_index("file")
totals   = stats_df.sum()

print("Filter attribution (all files combined):")
check_row_counts("  blank/null postcode filter",   int(totals["raw"]),                       int(totals["after_postcode_clean"]))
check_row_counts("  residential filter (D/S/T/F)", int(totals["after_postcode_clean"]),      int(totals["after_residential_filter"]))
check_row_counts("  ppd_category_type == 'A'",     int(totals["after_residential_filter"]),  int(totals["after_ppd_category_filter"]))
check_row_counts("  price >= 50,000",              int(totals["after_ppd_category_filter"]), int(totals["after_price_filter"]))

print("\nPer-file detail:")
display(stats_df)

### 2d. Concat + quality assessment

In [ ]:
hp = pd.concat(hp_dfs, ignore_index=True)
assess_quality(hp, "house prices post-concat")

### 2e. Deduplication on transaction_id

In [ ]:
before_dedup = len(hp)
hp = hp.drop_duplicates(subset="transaction_id", keep="last")
hp = hp.drop(columns="transaction_id").reset_index(drop=True)

check_row_counts("transaction_id dedup", before_dedup, len(hp))

### 2f. Category validation

In [ ]:
check_categories(hp, "property_type",     expected=RESIDENTIAL)
check_categories(hp, "ppd_category_type", expected={"A"})

### 2g. Price & date sanity checks

In [ ]:
print(f"Price  -- min: {hp['price'].min():,}   max: {hp['price'].max():,}")
print(f"Date   -- min: {hp['date_of_transfer'].min().date()}   max: {hp['date_of_transfer'].max().date()}")
print(f"Columns: {list(hp.columns)}")
display(hp.head())

### 2h. Write parquet

In [ ]:
write_parquet(hp, f"{SILVER_DIR}/houseprices_clean.parquet")

---
## Dataset 3: Postcode → LSOA Lookup

- **Source:** bronze Delta table `bronze_postcode_lookup` — two columns: `pcds`, `lsoa11`
- **Reporting grain:** Postcode (one row per unique postcode)
- **Output:** `data/silver/postcode_clean.parquet`
- **Columns:** `postcode`, `lsoa_code`

The ingestion layer already projected to only these two columns (from the 51-column ONSPD CSV).

### 3a. Load (two columns only)

In [ ]:
pc = read_bronze("bronze_postcode_lookup")[["pcds", "lsoa11"]].copy()
pc = standardise_columns(pc, rename_map={"pcds": "postcode", "lsoa11": "lsoa_code"})

raw_pc_count = len(pc)
print(f"Raw rows: {raw_pc_count:,}")
display(pc.head())

### 3b. Clean -- trim whitespace, drop null/blank, deduplicate on postcode

In [ ]:
pc["postcode"]  = pc["postcode"].str.strip()
pc["lsoa_code"] = pc["lsoa_code"].str.strip()

pc = pc.dropna(subset=["postcode", "lsoa_code"])
pc = pc[(pc["postcode"] != "") & (pc["lsoa_code"] != "")]
check_row_counts("drop null/blank rows", raw_pc_count, len(pc))

before_dedup = len(pc)
pc = pc.drop_duplicates(subset="postcode", keep="first").reset_index(drop=True)
check_row_counts("deduplicate on postcode", before_dedup, len(pc))

### 3c. Validate

In [ ]:
assess_quality(pc, "postcode_clean")
check_duplicates(pc, ["postcode"], "postcode_clean")
display(pc.head())

### 3d. Write parquet

In [ ]:
write_parquet(pc, f"{SILVER_DIR}/postcode_clean.parquet")

---
## Summary

All three silver parquet files have been written. Read back to confirm shapes and check key invariants.

In [ ]:
outputs = {
    "houseprices_clean.parquet": f"{SILVER_DIR}/houseprices_clean.parquet",
    "postcode_clean.parquet":    f"{SILVER_DIR}/postcode_clean.parquet",
    "claimant_clean.parquet":    f"{SILVER_DIR}/claimant_clean.parquet",
    "crime_clean.parquet":       f"{SILVER_DIR}/crime_clean.parquet",
    "health_clean.parquet":      f"{SILVER_DIR}/health_clean.parquet",
}

print("=== Silver layer outputs (cleaning_and_validation_layer) ===")
for name, path in outputs.items():
    df_check = pd.read_parquet(path)
    size_mb  = os.path.getsize(path) / 1_048_576
    print(f"  {name}: shape={df_check.shape}  size={size_mb:.1f} MB  cols={list(df_check.columns)}")

# Spot-check invariants
hp_check  = pd.read_parquet(f"{SILVER_DIR}/houseprices_clean.parquet")
pc_check  = pd.read_parquet(f"{SILVER_DIR}/postcode_clean.parquet")
cl_check  = pd.read_parquet(f"{SILVER_DIR}/claimant_clean.parquet")
cr_check  = pd.read_parquet(f"{SILVER_DIR}/crime_clean.parquet")
he_check  = pd.read_parquet(f"{SILVER_DIR}/health_clean.parquet")

assert set(hp_check["property_type"].unique()) <= RESIDENTIAL, "HP: non-residential type found"
assert pc_check["postcode"].is_unique,                         "Postcode: non-unique postcodes"
assert cl_check[["lsoa_code","year"]].duplicated().sum() == 0, "Claimant: duplicates at grain"
assert cr_check[["lsoa_code","year"]].duplicated().sum() == 0, "Crime: duplicates at grain"
assert he_check[["lsoa_code","year"]].duplicated().sum() == 0, "Health: duplicates at grain"

print("\nAll invariant checks passed.")
print("Note: adi_clean.parquet is produced by notebooks/featurengineering_transformation.ipynb")